In [ ]:
#import libraries
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import numpy as np
from tqdm.auto import tqdm        
import random

cols = ["No", "Time_Offset", "Type", "CAN_ID", "Data_Length", 'One', 'Two', 'Three', 'Four', 'Five', 'Six', 'Seven', 'Eight']

In [ ]:
simulation_df = pd.read_csv("KIA_SOUL_normal.csv")
simulation_df

Fuzz

In [ ]:
# DoS attack dataset generation
def fuzzy_df_gen():
    data_df = simulation_df[1:2].copy()
    
    temp = str(hex(random.randrange(0, 2048))[2:]).upper()
    for _ in range(4 - len(temp)):
        temp = '0' + temp    
    data_df["ID"] = temp

    data_l = random.randrange(1, 8)
    data_df["Data_Length"] = data_l
    for i in range(data_l):
        j = i + 5
        data_field = random.randrange(0, 255)
        temp = str(hex(data_field)[2:]).upper()
        for _ in range(2 - len(temp)):
            temp = '0' + temp
        data_df[cols[j]] = temp
    data_df["Label"] = "Fuzzy"
    return data_df

In [ ]:
temp = str(hex(random.randrange(0, 2048))[2:]).upper()
for _ in range(4 - len(temp)):
    temp = '0' + temp

In [ ]:
def inject_attack_dataset(orig_df, attack_gen_func, time_gap_range=(10, 50), label="Attack", start_time=0.0):
    end_time = float(orig_df.iloc[-1]["Time_Offset"])
    attack_messages = []
    current_time = start_time

    min_gap, max_gap = time_gap_range
    last_pct = -1

    while current_time < end_time:
        # Generate one attack message
        attack_row = attack_gen_func().copy()
        attack_row["Time_Offset"] = current_time
        attack_row["Label"] = label
        attack_messages.append(attack_row)

        # Update time
        time_gap = random.uniform(min_gap, max_gap)
        current_time += time_gap

        # Progress printing
        pct = int(current_time / end_time * 100)
        if pct != last_pct:
            print(f"\rInjecting '{label}' messages… {pct:3d}% complete", end="")
            last_pct = pct

    print(f"\nDone – injected {len(attack_messages)} {label} messages.")

    # Merge and sort
    attack_df = pd.concat(attack_messages)
    merged_df = pd.concat([orig_df, attack_df], axis=0)
    merged_df["Time_Offset"] = merged_df["Time_Offset"].astype(float)
    df_sort_time = merged_df.sort_values(by='Time_Offset')

    return df_sort_time[1:]


In [ ]:
df_with_fuzzy = inject_attack_dataset(
    orig_df=simulation_df,
    attack_gen_func=fuzzy_df_gen,
    time_gap_range=(10, 50),
    label="Fuzzy"
)


DoS

In [ ]:
# DoS attack dataset generation
def dos_df_gen():
    data_df = simulation_df[1:2].copy()
    
    data_df["ID"] = "0000"
    data_df["Data_Length"] 
    
    for i in range(8):
        j = i + 5
        data_field = i
        temp = str(hex(data_field)[2:]).upper()
        for _ in range(2 - len(temp)):
            temp = '0' + temp
        data_df[cols[j]] = temp
    data_df["Label"] = "DoS"
    return data_df

In [ ]:
temp = str(hex(random.randrange(0, 2048))[2:]).upper()
for _ in range(4 - len(temp)):
    temp = '0' + temp

In [ ]:
df_with_fuzzy = inject_attack_dataset(
    orig_df=simulation_df,
    attack_gen_func=fuzzy_df_gen,
    time_gap_range=(10, 50),
    label="Fuzzy"
)


Replay

In [ ]:
def inject_sequential_from_df(orig_df, injection_df, time_gap_range=(10, 50), label="Attack", start_time=0.0):
    end_time = float(orig_df.iloc[-1]["Time_Offset"])
    attack_rows = []
    current_time = start_time

    min_gap, max_gap = time_gap_range
    last_pct = -1

    injection_index = 0
    total_injections = 0
    injection_len = len(injection_df)

    while current_time < end_time:
        # Get next injection row in order (loop if needed)
        attack_row = injection_df.iloc[[injection_index]].copy()
        attack_row["Time_Offset"] = current_time
        attack_row["Label"] = label
        attack_rows.append(attack_row)

        # Update index and time
        injection_index = (injection_index + 1) % injection_len
        time_gap = random.uniform(min_gap, max_gap)
        current_time += time_gap
        total_injections += 1

        # Progress
        pct = int(current_time / end_time * 100)
        if pct != last_pct:
            print(f"\rSequential injection… {pct:3d}% complete", end="")
            last_pct = pct

    print(f"\nDone – injected {total_injections} sequential messages.")

    # Merge and sort
    attack_df = pd.concat(attack_rows, ignore_index=True)
    merged_df = pd.concat([orig_df, attack_df], axis=0)
    merged_df["Time_Offset"] = merged_df["Time_Offset"].astype(float)
    df_sort_time = merged_df.sort_values(by='Time_Offset')

    return df_sort_time


In [ ]:
df_with_seq_attacks = inject_sequential_from_df(
    orig_df=simulation_df,
    injection_df=simulation_df,
    time_gap_range=(5, 15),  
    label="Replay"
)


In [ ]:
import pandas as pd 
import numpy as np 
from DataGen import FuzzInjection

df_path = '/home/lisa/Arupreza/UIDS-II/Input_data/'

In [ ]:
simulation_df = pd.read_csv(df_path + "Kia_AF.csv")
time_gap_range = (10, 50)
num_attacks = 200000
label = "Fuzz"

Kia_Fuzz_H = FuzzInjection(time_gap_range, num_attacks, simulation_df, label)

Injecting 'Fuzz' messages…  42% complete

In [21]:
Kia_Fuzz_H[:30]

,Time_Offset,CAN_ID,Data_Length,One,Two,Three,Four,Five,Six,Seven,Eight,Label
600000,0.000000,0251,6,A9,81,2B,63,07,8D,A7,7E,Fuzz
600001,1.140624,0251,2,B3,81,C7,15,00,D2,A7,7E,Fuzz
600002,2.281247,0251,2,E2,21,C7,15,00,D2,A7,7E,Fuzz
600003,3.421870,0251,1,99,23,C7,15,00,D2,A7,7E,Fuzz
600004,4.562494,0251,1,F0,23,C7,15,00,D2,A7,7E,Fuzz
600005,5.703118,0251,6,10,07,68,9B,5B,05,A7,7E,Fuzz
0,6.000000,0340,8,0F,00,D4,2B,EC,00,58,1A,Normal
1,6.300000,0251,8,C5,23,C7,15,00,D2,A7,7E,Normal
2,6.400000,02B0,6,EF,FF,00,07,17,D0,-1,-1,Normal
600006,6.843741,0251,4,8F,7A,66,81,00,D2,A7,7E,Fuzz
